# 🎵 Non-Gunshot Trimmer — Clean Background Audio Extraction

Extracts **clean non-gunshot clips** from background audio sources. Strict impulse rejection ensures no gunshot-like sounds leak in.

## What This Notebook Does
1. Scans `Data/sound/` and optionally `Data/audio/`
2. Uses sliding window (250ms, 125ms hop = 50% overlap) to extract clips
3. **Impulse rejection**: Skips any window that looks suspiciously impulsive
4. **Silence rejection**: Skips dead silence
5. **Diversity**: Caps clips per file, ensures variety
6. Every output clip is **exactly 250ms** (5,512 samples @ 22,050 Hz)

## Output
- `Data/TRIMMED_NONGUNSHOTS/verified/` ← All confirmed non-gunshot clips
- `Data/TRIMMED_NONGUNSHOTS/suspicious/` ← Impulse-rejected clips (for review)
- `Data/TRIMMED_NONGUNSHOTS/reports/` ← manifest.csv + summary.json

See `manual/README.md` for full documentation.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
%pip install -q librosa soundfile pandas numpy tqdm matplotlib

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import librosa
import librosa.display
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import json
import shutil
import random
import re
import IPython.display as ipd

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print('✅ All imports loaded successfully.')

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION — CHANGE PATHS HERE
# ============================================================

# --- Path Auto-Detection ---
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# !! TO HARDCODE: Uncomment the line below !!
# PROJECT_ROOT = Path(r'd:\Desktop\Data-Cleaner')

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = DATA_DIR / 'TRIMMED_NONGUNSHOTS'

# --- Audio Parameters ---
SAMPLE_RATE = 22050
TARGET_MS = 250
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_MS / 1000)  # = 5512

CLASS0_HOP_MS = 125            # Sliding window hop (50% overlap)
CLASS0_HOP_SAMPLES = int(SAMPLE_RATE * CLASS0_HOP_MS / 1000)

MAX_CLIPS_PER_FILE = 40        # Diversity cap: max clips from one source file

# --- Source Directories ---
CLASS0_DIRS = [DATA_DIR / 'sound']       # Always included
INCLUDE_AUDIO_FOLDER = True               # Set True to also include Data/audio/
if INCLUDE_AUDIO_FOLDER:
    CLASS0_DIRS.append(DATA_DIR / 'audio')

# --- Impulse Rejection Thresholds ---
PEAK_SILENCE = 0.003           # Below this = silence
CREST_IMPULSE = 8.5            # Above this + high peak = suspicious
PEAK_IMPULSE = 0.1             # Must also have high peak to trigger
CENTROID_IMPULSE = 5000.0      # High centroid + high attack = suspicious
ATTACK_RATIO_IMPULSE = 10.0    # Attack ratio threshold for spectral guard

# --- Clean previous output? ---
OVERWRITE = True

# --- Print config ---
print(f'Project Root      : {PROJECT_ROOT}')
print(f'Data Directory    : {DATA_DIR}')
print(f'Output            : {OUTPUT_DIR}')
print(f'Clip Duration     : {TARGET_MS}ms = {TARGET_SAMPLES} samples @ {SAMPLE_RATE}Hz')
print(f'Hop               : {CLASS0_HOP_MS}ms (50% overlap)')
print(f'Max Clips/File    : {MAX_CLIPS_PER_FILE}')
print(f'Include audio/    : {INCLUDE_AUDIO_FOLDER}')
print(f'Source dirs       : {[str(d) for d in CLASS0_DIRS]}')
assert DATA_DIR.exists(), f'❌ Data directory not found: {DATA_DIR}'

In [ ]:
# ============================================================
# CELL 4: Helper Functions
# ============================================================
JUNK_TOKENS = ('__MACOSX',)
SAFE_NAME_RE = re.compile(r'[^A-Za-z0-9._-]+')


def is_junk(path):
    as_str = str(path)
    return any(t in as_str for t in JUNK_TOKENS) or path.name.startswith('._')


def sanitize_name(raw, max_len=120):
    cleaned = SAFE_NAME_RE.sub('_', raw.strip()).strip('._')
    return (cleaned or 'clip')[:max_len]


def collect_wavs(dirs):
    files = []
    for d in dirs:
        if d.exists():
            files.extend([f for f in d.rglob('*.wav') if not is_junk(f)])
    return sorted(files)


def load_audio(path):
    y, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y.astype(np.float32)


def force_exact_length(clip):
    if len(clip) == TARGET_SAMPLES:
        return clip
    if len(clip) > TARGET_SAMPLES:
        return clip[:TARGET_SAMPLES]
    return np.pad(clip, (0, TARGET_SAMPLES - len(clip)), mode='constant')


def normalize_clip(clip):
    clip = clip - np.mean(clip)
    peak = float(np.max(np.abs(clip))) if len(clip) else 0.0
    if peak > 0.999:
        clip = clip / peak * 0.999
    return clip.astype(np.float32)


def write_clip(path, clip):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), clip, SAMPLE_RATE, subtype='PCM_16')
    if not path.exists() or path.stat().st_size <= 44:
        raise IOError(f'Empty/invalid file written: {path}')


print('✅ Helper functions defined.')

In [ ]:
# ============================================================
# CELL 5: Background Clip Extraction (Sliding Window)
# ============================================================

def extract_background_windows(y):
    """
    Extract 250ms windows from a background audio file using a sliding window
    with 50% overlap. Returns list of (clip, start_sample, end_sample).
    
    Windows are shuffled to randomize selection when capping per file.
    """
    if y is None or len(y) < 100:
        return []
    
    windows = []
    
    # If file is shorter than one clip, pad it
    if len(y) < TARGET_SAMPLES:
        clip = force_exact_length(y)
        clip = normalize_clip(clip)
        windows.append((clip, 0, len(y)))
        return windows
    
    # Generate all valid start positions
    starts = list(range(0, len(y) - TARGET_SAMPLES + 1, CLASS0_HOP_SAMPLES))
    random.shuffle(starts)  # Randomize for diversity
    
    for start_idx in starts:
        end_idx = start_idx + TARGET_SAMPLES
        clip = y[start_idx:end_idx]
        clip = force_exact_length(clip)  # Safety guarantee
        clip = normalize_clip(clip)
        windows.append((clip, start_idx, end_idx))
        
        if len(windows) >= MAX_CLIPS_PER_FILE:
            break
    
    return windows


print('✅ Background extraction function defined.')

In [ ]:
# ============================================================
# CELL 6: Safety Checks (Impulse + Silence + Spectral Guard)
# ============================================================

def nongunshot_safety_check(clip):
    """
    Check if a background clip is safe (no gunshot-like characteristics).
    
    Returns: (verdict: str, metrics: dict)
        verdict: 'ok', 'silence', 'impulse', 'spectral_suspect'
    """
    abs_clip = np.abs(clip)
    peak = float(np.max(abs_clip)) if len(abs_clip) else 0.0
    rms = float(np.sqrt(np.mean(np.square(clip)))) if len(clip) else 0.0
    crest = peak / (rms + 1e-12)
    median_abs = float(np.median(abs_clip)) if len(abs_clip) else 0.0
    prominence = peak / (median_abs + 1e-12)
    
    # Attack ratio
    attack = float(np.max(np.abs(np.diff(clip)))) if len(clip) > 1 else 0.0
    attack_ratio = attack / (rms + 1e-12)
    
    # Spectral centroid
    try:
        centroid = float(np.mean(librosa.feature.spectral_centroid(
            y=clip, sr=SAMPLE_RATE)[0]))
    except Exception:
        centroid = 0.0
    
    metrics = {
        'peak': peak,
        'rms': rms,
        'crest_factor': crest,
        'prominence': prominence,
        'attack_ratio': attack_ratio,
        'spectral_centroid': centroid,
        'median_abs': median_abs,
    }
    
    # --- Safety checks ---
    # 1. Silence check
    if peak < PEAK_SILENCE or rms < 0.0005:
        return 'silence', metrics
    
    # 2. Impulse rejection (high crest + high peak = possible gunshot)
    if crest > CREST_IMPULSE and peak > PEAK_IMPULSE:
        return 'impulse', metrics
    
    # 3. High prominence + high attack = suspicious transient
    if prominence > 15.0 and attack_ratio > ATTACK_RATIO_IMPULSE:
        return 'impulse', metrics
    
    # 4. Spectral guard: very high centroid + high attack
    if centroid > CENTROID_IMPULSE and attack_ratio > ATTACK_RATIO_IMPULSE:
        return 'spectral_suspect', metrics
    
    return 'ok', metrics


print('✅ Safety check function defined.')

In [ ]:
# ============================================================
# CELL 7: BUILD PIPELINE — Run the full extraction
# ============================================================

def build_nongunshot_dataset():
    """Main pipeline: extract, validate, and write all non-gunshot clips."""
    
    # --- Setup output directories ---
    if OUTPUT_DIR.exists() and OVERWRITE:
        shutil.rmtree(OUTPUT_DIR)
    
    verified_dir = OUTPUT_DIR / 'verified'
    suspicious_dir = OUTPUT_DIR / 'suspicious'
    reports_dir = OUTPUT_DIR / 'reports'
    
    verified_dir.mkdir(parents=True, exist_ok=True)
    suspicious_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    # --- Collect source files ---
    source_files = collect_wavs(CLASS0_DIRS)
    random.shuffle(source_files)  # Randomize source order for diversity
    
    print(f'\n{"=" * 65}')
    print(f'NON-GUNSHOT TRIMMER — Starting Build')
    print(f'{"=" * 65}')
    print(f'Source files found  : {len(source_files):,}')
    print(f'Output directory    : {OUTPUT_DIR}')
    print(f'Clip duration       : {TARGET_MS}ms ({TARGET_SAMPLES} samples)')
    print(f'Hop                 : {CLASS0_HOP_MS}ms (50% overlap)')
    print(f'Max clips per file  : {MAX_CLIPS_PER_FILE}')
    print(f'{"=" * 65}\n')
    
    manifest_rows = []
    load_errors = []
    verified_count = 0
    suspicious_count = 0
    silence_skipped = 0
    skipped_files = 0
    clip_index = 0
    
    for src_path in tqdm(source_files, desc='🎵 Extracting backgrounds', unit='file'):
        src_stem = sanitize_name(src_path.stem)
        src_parent = sanitize_name(src_path.parent.name)
        source_key = f'{src_parent}_{src_stem}'
        
        # Load audio
        try:
            y = load_audio(src_path)
        except Exception as exc:
            load_errors.append({'source_path': str(src_path), 'reason': f'load_error: {exc}'})
            skipped_files += 1
            continue
        
        if len(y) < 4:
            skipped_files += 1
            continue
        
        # Extract windows
        windows = extract_background_windows(y)
        
        for w_idx, (clip, src_start, src_end) in enumerate(windows):
            verdict, metrics = nongunshot_safety_check(clip)
            
            if verdict == 'ok':
                clip_index += 1
                out_name = f'c0_{clip_index:07d}_{source_key}_w{w_idx:03d}.wav'
                out_path = verified_dir / out_name
                write_clip(out_path, clip)
                verified_count += 1
                
                manifest_rows.append({
                    'output_path': str(out_path.relative_to(OUTPUT_DIR)),
                    'source_path': str(src_path),
                    'source_dir': src_parent,
                    'label': 0,
                    'window_index': w_idx,
                    'start_sample': src_start,
                    'end_sample': src_end,
                    'duration_ms': TARGET_MS,
                    'sample_rate': SAMPLE_RATE,
                    'decision': 'accepted',
                    **metrics,
                })
            
            elif verdict == 'silence':
                silence_skipped += 1
            
            else:  # 'impulse' or 'spectral_suspect'
                suspicious_count += 1
                sus_name = f'sus_{suspicious_count:07d}_{source_key}_w{w_idx:03d}_{verdict}.wav'
                sus_path = suspicious_dir / sus_name
                write_clip(sus_path, clip)
                
                manifest_rows.append({
                    'output_path': str(sus_path.relative_to(OUTPUT_DIR)),
                    'source_path': str(src_path),
                    'source_dir': src_parent,
                    'label': 'suspicious',
                    'window_index': w_idx,
                    'start_sample': src_start,
                    'end_sample': src_end,
                    'duration_ms': TARGET_MS,
                    'sample_rate': SAMPLE_RATE,
                    'decision': verdict,
                    **metrics,
                })
    
    # --- Save reports ---
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(reports_dir / 'manifest.csv', index=False)
    
    if load_errors:
        pd.DataFrame(load_errors).to_csv(reports_dir / 'skipped_sources.csv', index=False)
    
    summary = {
        'total_source_files': len(source_files),
        'skipped_source_files': skipped_files,
        'verified_clips': verified_count,
        'suspicious_clips': suspicious_count,
        'silence_skipped': silence_skipped,
        'config': {
            'sample_rate': SAMPLE_RATE,
            'target_ms': TARGET_MS,
            'hop_ms': CLASS0_HOP_MS,
            'max_clips_per_file': MAX_CLIPS_PER_FILE,
            'include_audio_folder': INCLUDE_AUDIO_FOLDER,
            'crest_impulse': CREST_IMPULSE,
            'peak_impulse': PEAK_IMPULSE,
        },
    }
    (reports_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    
    print(f'\n{"=" * 65}')
    print(f'BUILD COMPLETE')
    print(f'{"=" * 65}')
    print(f'✅ Verified non-gunshot clips : {verified_count:,}')
    print(f'⚠️  Suspicious clips           : {suspicious_count:,}')
    print(f'🔇 Silence skipped             : {silence_skipped:,}')
    print(f'⏭️  Skipped source files        : {skipped_files:,}')
    print(f'📁 Output                      : {OUTPUT_DIR}')
    print(f'{"=" * 65}')
    
    return manifest_df


# 🚀 RUN THE BUILD
manifest_df = build_nongunshot_dataset()

In [ ]:
# ============================================================
# CELL 8: VERIFICATION
# ============================================================
print('Verifying all output files...\n')

verified_files = list((OUTPUT_DIR / 'verified').glob('*.wav'))
suspicious_files = list((OUTPUT_DIR / 'suspicious').glob('*.wav'))

bad_files = []
for f in tqdm(verified_files + suspicious_files, desc='Checking sizes'):
    y, sr = librosa.load(f, sr=None)
    if len(y) != TARGET_SAMPLES:
        bad_files.append((f.name, len(y)))

if bad_files:
    print(f'\n⚠️ WARNING: {len(bad_files)} files are NOT exactly {TARGET_SAMPLES} samples!')
    for name, length in bad_files[:10]:
        print(f'  {name}: {length} samples')
else:
    total = len(verified_files) + len(suspicious_files)
    print(f'\n✅ ALL {total:,} files are exactly {TARGET_SAMPLES} samples ({TARGET_MS}ms). Perfect!')

print(f'\n📊 Summary:')
print(f'  Verified clips    : {len(verified_files):,}')
print(f'  Suspicious clips  : {len(suspicious_files):,}')

# Source diversity check
if not manifest_df.empty:
    accepted = manifest_df[manifest_df['decision'] == 'accepted']
    if not accepted.empty:
        source_counts = accepted['source_dir'].value_counts()
        print(f'\n🌍 Source diversity ({len(source_counts)} unique source folders):')
        for src, count in source_counts.head(10).items():
            print(f'  {src:<40} : {count:,} clips')

In [ ]:
# ============================================================
# CELL 9: VISUAL AUDIT
# ============================================================
if not verified_files:
    print('No verified files to plot.')
else:
    n_samples = min(6, len(verified_files))
    sample_files = random.sample(verified_files, n_samples)
    
    fig, axes = plt.subplots(n_samples, 2, figsize=(14, 3 * n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i, f in enumerate(sample_files):
        y, sr = librosa.load(f, sr=None)
        
        librosa.display.waveshow(y, sr=sr, ax=axes[i, 0])
        axes[i, 0].set_title(f.name[:50], fontsize=9)
        
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[i, 1])
        axes[i, 1].set_title('Mel-Spectrogram', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'reports' / 'visual_audit.png', dpi=150)
    plt.show()
    
    print('\n🔊 Listen to verified clips:')
    for f in sample_files[:3]:
        print(f'  {f.name}')
        display(ipd.Audio(f))

In [ ]:
# ============================================================
# CELL 10: BALANCE CHECK vs Gunshot Trimmer
# ============================================================
gunshot_dir = DATA_DIR / 'TRIMMED_GUNSHOTS' / 'verified'

if gunshot_dir.exists():
    gunshot_count = len(list(gunshot_dir.glob('*.wav')))
    nongunshot_count = len(verified_files)
    
    print(f'\n{"=" * 50}')
    print(f'BALANCE CHECK')
    print(f'{"=" * 50}')
    print(f'🔫 Gunshot clips     : {gunshot_count:,}')
    print(f'🎵 Non-gunshot clips : {nongunshot_count:,}')
    
    if nongunshot_count >= gunshot_count:
        ratio = nongunshot_count / max(gunshot_count, 1)
        print(f'\n✅ Ratio: 1 : {ratio:.2f} (non-gunshot has enough clips)')
        if ratio > 1.5:
            print(f'💡 TIP: You have {ratio:.1f}x more non-gunshot clips. '
                  f'Consider downsampling to {gunshot_count:,} for perfect 1:1 balance.')
    else:
        print(f'\n⚠️ Non-gunshot has FEWER clips than gunshot!')
        print(f'   Consider enabling INCLUDE_AUDIO_FOLDER = True for more sources.')
else:
    print('\nℹ️ Gunshot Trimmer has not been run yet (Data/TRIMMED_GUNSHOTS/verified/ not found).')
    print('   Run Trimmer 01 first, then re-run this cell for a balance check.')